# Evaluación del Riesgo Operacional en Canales Electrónicos


En este espacio de trabajo se plantea como caso de estudio la evaluación del riesgo operacional en los canales electrónicos de una entidad financiera líder, con el objetivo de transformar datos históricos de eventos de riesgo tecnológico en una herramienta que facilite la toma de decisiones gerenciales y la optimización de los controles existentes.

La entidad ha recibido una base de datos que contiene información sobre eventos de riesgo en distintos canales electrónicos, como aplicaciones móviles, sucursales virtuales y cajeros automáticos. A partir de esta información, se busca identificar la exposición económica al riesgo y evaluar la eficacia de los controles implementados en dichos canales.

El proceso de análisis se desarrolla bajo la metodología de riesgo operacional, iniciando con la construcción de una arquitectura de matrices integrada. En primer lugar, se elabora la matriz de frecuencia, que permite contabilizar el número de ocurrencias de los eventos. Posteriormente, se construye la matriz de severidad, en la cual se clasifican los eventos según el nivel de impacto o magnitud del daño generado.

A partir de estas dos matrices, se desarrolla la matriz de pérdidas agregadas, calculada como el producto entre la frecuencia y la severidad, lo que permite estimar el impacto económico total de los eventos de riesgo. De manera complementaria, se construye la matriz de impacto y gestión, que facilita el análisis de los riesgos en función de su tratamiento y control.

En la fase de estimación cuantitativa, se calcula la Pérdida Esperada (PE) como medida base del riesgo. Asimismo, se estima el Operational Value at Risk (OpVaR) con un nivel de confianza del 99,9%, con el fin de evaluar posibles pérdidas extremas. Este análisis se realiza considerando dos escenarios: pérdidas no gestionadas, que representan el riesgo inherente, y pérdidas gestionadas, que reflejan el riesgo residual tras la aplicación de controles.

Adicionalmente, se diseña una herramienta de auditoría basada en un filtro dinámico, que permite seleccionar cualquier celda de la matriz de pérdidas y visualizar de manera automática el detalle de los eventos asociados, junto con su descripción. Esta funcionalidad facilita la trazabilidad de la información y fortalece el proceso de análisis y supervisión del riesgo.

Finalmente, los resultados obtenidos permiten a la entidad comprender mejor su exposición al riesgo operacional en canales electrónicos, identificar oportunidades de mejora en los controles y tomar decisiones estratégicas orientadas a mitigar pérdidas y fortalecer la gestión del riesgo.

0. Se procede con la carga de las librerias de trabajo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

0. Creamos la funcion de clusterizacion para poderla reutilzar varias veces

In [ ]:
def clusterizacion(Xi):

  # Proceso de clusterizacion
  XC=np.random.choice(Xi,size=5)
  XC=np.sort(XC) # Semillas de la Clusterizacion (Improbable, Posible, Ocasional, Probable, Frecuente)
  nc=np.zeros((len(Xi),1)) # Aqui clasifico los datos en cada una de las semillas

  for k in range(len(Xi)):
    d=np.abs(XC-Xi[k]) # Distacia de un dato a cada semilla
    nc[k]=np.argmin(d)
    nc2=np.int32(nc[k]) # Numero de pertenencia al cluster en enteros
    XC[nc2,]=(XC[nc2,]+Xi[k])/2 # Actualizamos la semilla a la que pertenece cada dato (K-Medoids)

  return XC,nc # Me retorna clusters y cluster al que pertenece un dato

1. Se cargan los archivos de trabajo

In [ ]:
from re import X
nxl='/content/drive/MyDrive/integración de datos y prospectiva/Reto 5/5. Riesgo Operacional FallasTecnológicas.xlsx'
XDB=pd.read_excel(nxl,sheet_name=0)

# La primera fila del archivo Excel contiene los nombres de las columnas, así que se usa para establecer las cabeceras
XDB.columns = XDB.iloc[0]
# Elimina la primera fila (que ahora es el encabezado) y reinicia el índice del DataFrame
XDB = XDB[1:].reset_index(drop=True)

# Limpia los nombres de las columnas eliminando espacios en blanco al inicio y al final
XDB.columns = XDB.columns.str.strip()

# Renombra algunas columnas para mayor claridad y consistencia
XDB.rename(columns={
    'Fechas': 'Fecha', # Renombra 'Fechas' a 'Fecha'
    'Transacciones Diarias': 'Transacciones Diarias', # Mantiene 'Transacciones Diarias'
    'Transacciones Fallidas': 'Frecuencia_original', # Renombra 'Transacciones Fallidas' a 'Frecuencia_original'
    'Valor Generado Promedio (Millones)': 'Severidad_original', # Renombra 'Valor Generado Promedio (Millones)' a 'Severidad_original'
    'Descripción Evento': 'Riesgos_descripcion' # Renombra 'Descripción Evento' a 'Riesgos_descripcion'
}, inplace=True)

# Convierte las columnas de frecuencia y severidad a tipo numérico, manejando errores con 'coerce' (convierte valores no numéricos a NaN)
XDB['Frecuencia_original'] = pd.to_numeric(XDB['Frecuencia_original'], errors='coerce')
XDB['Severidad_original'] = pd.to_numeric(XDB['Severidad_original'], errors='coerce')

# Elimina las filas que tienen valores NaN (Not a Number) en las columnas de frecuencia o severidad
XDB.dropna(subset=['Frecuencia_original', 'Severidad_original'], inplace=True)
# Reinicia el índice del DataFrame después de eliminar filas
XDB.reset_index(drop=True, inplace=True)

# Establece la semilla para la generación de números aleatorios, asegurando reproducibilidad
np.random.seed(42)

# Extrae la variable de Frecuencia y la convierte en un array de NumPy
Xf=np.array(XDB['Frecuencia_original'])
XCf,ncf=clusterizacion(Xf)

# Extrae la variable de Severidad y la convierte en un array de NumPy
Xs=np.array(XDB['Severidad_original'])
XCs,ncs=clusterizacion(Xs)

# Calcula las Pérdidas Agregadas (LDA) como el producto de Frecuencia y Severidad
LDA=Xf*Xs
XClda,nclda=clusterizacion(LDA)

# Crea una copia del DataFrame XDB para el DataFrame final (df)
df = XDB.copy()
df['Freq'] = Xf
df['Nivel_f'] = ncf
df['Sev'] = Xs
df['Nivel_S'] = ncs
df['LDA'] = LDA
df['Nivel_LDA'] = nclda

# Renombra la columna 'Riesgos_descripcion' a 'Riesgos' para consistencia
df.rename(columns={'Riesgos_descripcion': 'Riesgos'}, inplace=True)

# Muestra las primeras filas del DataFrame df para verificar los cambios
df.head()

2. Procedemos con la construccion de la matriz de eventos de riesgo

In [ ]:
MEf=np.zeros((5,5)) # Matriz frecuencia
MEs=np.zeros((5,5)) # Matriz severidad

for k in range(len(Xf)):
  nf=np.int32(df.loc[k,'Nivel_f'])   # Fila de frecuencia
  nc=np.int32(df.loc[k,'Nivel_S'])   # Columna de impacto
  MEf[nf,nc]=MEf[nf,nc]+Xf[k,]
  MEs[nf,nc]=(MEs[nf,nc]+Xs[k,])/2 # Costo promedio

plt.figure()
sns.heatmap(MEf,annot=True,fmt='.0f',cmap='jet_r')
plt.title('Matriz Eventos')
plt.show()

plt.figure()
sns.heatmap(MEs,annot=True,fmt='.3f',cmap='jet_r')
plt.title('Matriz Impacto')
plt.show()

MLDA = MEf * MEs # Matriz de Pérdidas Agregadas

plt.figure()
sns.heatmap(MLDA, annot=True, fmt='.3f', cmap='jet_r')
plt.title('Matriz de Pérdidas Agregadas')
plt.show()

In [ ]:
# Procedemos con la creacion de la matriz de impacto
MI=np.array(([1,1,1,2,2],
            [1,2,2,3,3],
            [1,2,3,3,4],
            [2,3,3,4,4],
            [2,3,4,4,5]))

plt.figure()
sns.heatmap(MI,annot=True,fmt='.0f',cmap='jet')
plt.title('Matriz de Impacto(Actividades de gestion)')
plt.show()

In [ ]:
# Matriz de Gestion - Involucra recursos de la Organizacion
NG=np.int32(input("Ingresar el Nivel de Gestion (1,2,3,4):"))
MG=np.copy(MI)

for i in range(5): # Si es mayor que 1 y menor que 5 incremente la gestion
  for j in range(5):
    if MG[i,j]>1 and MG[i,j]<5:
      MG[i,j]=MG[i,j]*NG

plt.figure()
sns.heatmap(MG,annot=True,fmt='.0f',cmap='jet')
plt.title('Matriz de Gestion(Actividades de gestion)')
plt.show()

3. Estimación Cuantitativa

In [ ]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

np.random.seed(42)
df["gestionado"] = np.random.choice(["Gestionado", "No Gestionado"], size=len(df))

In [ ]:
#1. Perdida Esperada por Promedio Simple
ps=np.mean(MEs)
print("El promedio simple de severidad por atender una falla es:",ps)

#2. Perdida Esperada por Promedio Ponderado
pp=np.sum(MEf*MEs)/np.sum(MEf)
print("El promedio ponderado de severidad por atender una falla es:",pp)

#3. Se procede a determinar la perdida ponderada teniendo igualmente la matriz de Impacto - Nivel de Riesgo
pnr=np.sum(MEf*MEs*MI)/np.sum(MEf*MI)
print("El promedio ponderado de severidad por atender una falla es:",pnr)

#4. Se procede a determinar la perdida ponderada teniendo igualmente la matriz de Gestion
png=np.sum(MEf*MEs*MI)/np.sum(MEf*MG)
print(f"El promedio ponderado de severidad por atender una falla con un nivel de gestion {NG} es:",png)

#5. Operational Value at Risk (OpVaR)
OpVaR = np.percentile(df["LDA"], 99.9)
print("El valor operativo en riesgo a un nivel de confianza del 99.9% es:", OpVaR)

no_gestionado = df[df["gestionado"] == "No Gestionado"]["LDA"]
gestionado = df[df["gestionado"] == "Gestionado"]["LDA"]

print("El valor operativo con riesgo inherente es:", np.percentile(no_gestionado, 99.9))
print("El valor operativo con riesgo residual es:", np.percentile(gestionado, 99.9))

**Análisis de Resultados**
El promedio simple de severidad por atender una falla es de aproximadamente 4.05, lo que indica que, en términos generales, cada evento de riesgo tiene un impacto relevante sobre la operación. Sin embargo, al analizar el promedio ponderado, se observa que el costo real esperado por evento se reduce a valores entre 2.17 y 2.81, lo que evidencia que los eventos más frecuentes tienden a ser de menor severidad, reduciendo el impacto promedio efectivo.

De acuerdo con la integración de la matriz de impacto, se identifica que la pérdida esperada puede variar dependiendo de la relevancia de los eventos dentro de la matriz de frecuencia, concentrándose el riesgo en aquellos eventos menos frecuentes pero más severos. Al incorporar niveles de gestión, específicamente en un nivel de gestión 2, la severidad ponderada se reduce a aproximadamente 1.51, lo que demuestra que la implementación de controles tiene un efecto significativo en la mitigación del riesgo.

Por otro lado, el Valor en Riesgo Operacional (OpVaR) al 99.9% se estima en 102.90, reflejando la exposición a eventos extremos. Al comparar el riesgo inherente (100.48) con el riesgo residual (94.15), se evidencia una reducción gracias a la gestión del riesgo, aunque esta disminución no es lo suficientemente amplia como para eliminar completamente la exposición a pérdidas severas.

**Matriz de Impacto:**
Esta matriz permite identificar el nivel de impacto que tienen los diferentes eventos de riesgo sobre las pérdidas de la organización, destacando aquellos eventos que, aunque poco frecuentes, pueden generar consecuencias económicas significativas.

**Matriz de Gestión:**
Esta matriz complementa la matriz de impacto al establecer el nivel de atención que requiere cada tipo de evento de riesgo. A mayor nivel de gestión, mayor es la intervención y los recursos asignados para mitigar el riesgo. Es importante considerar que la gestión implica costos operativos; sin embargo, su correcta implementación contribuye a reducir la probabilidad de pérdidas catastróficas y a disminuir el riesgo residual de la organización.


4. Herramienta de Auditoría (Filtro Dinámico)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def mostrar_detalle_MLDA(nivel_f, nivel_s):

    filtro = df[
        (df["Nivel_f"] == nivel_f) &
        (df["Nivel_S"] == nivel_s)
    ]

    if filtro.empty:
        print("No hay eventos en esta celda de MLDA")
    else:
        print(f"Eventos encontrados: {len(filtro)}\n")

        display(filtro[[
            "Fecha",
            "Transacciones Diarias",
            "Frecuencia_original",
            "Severidad_original",
            "LDA",
            "Riesgos"
        ]])

frecuencia_level_selector = widgets.Dropdown(
    options=np.sort(df["Nivel_f"].unique()),
    description="Nivel Frecuencia:"
)

severidad_level_selector = widgets.Dropdown(
    options=np.sort(df["Nivel_S"].unique()),
    description="Nivel Severidad:"
)

widgets.interactive(
    mostrar_detalle_MLDA,
    nivel_f=frecuencia_level_selector,
    nivel_s=severidad_level_selector
)

**Conclusión – Herramienta de Auditoría de la Matriz de Pérdidas**

La implementación de la herramienta de auditoría basada en el filtro dinámico de la matriz de pérdidas permite transformar el análisis de riesgo operacional en un proceso más transparente, trazable y orientado a la toma de decisiones. A través de esta funcionalidad, es posible seleccionar cualquier combinación de evento y nivel de severidad, y acceder de manera inmediata al detalle de los registros que componen dicha celda, incluyendo variables operativas y la pérdida estimada.

Este enfoque facilita la identificación de patrones específicos dentro de los eventos de riesgo, permitiendo distinguir cuáles son los principales generadores de pérdidas y cómo se distribuyen en el tiempo. Asimismo, la herramienta fortalece la capacidad de validación y control, ya que permite verificar la consistencia de los resultados agregados frente a los datos originales, reduciendo la opacidad en el análisis.

Adicionalmente, la posibilidad de profundizar en cada segmento de la matriz contribuye a mejorar la gestión del riesgo, al enfocar los esfuerzos en eventos críticos y en aquellos casos donde los controles no están siendo suficientemente efectivos. En este sentido, la herramienta no solo cumple una función descriptiva, sino que se convierte en un apoyo clave para la toma de decisiones estratégicas y la optimización de los mecanismos de mitigación.

En conclusión, la herramienta de auditoría añade valor significativo al modelo de riesgo operacional, al integrar análisis cuantitativo con capacidad de exploración detallada, permitiendo una gestión más precisa, informada y proactiva frente a los riesgos identificados.
